In [52]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression, LogisticRegression

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

from imblearn.over_sampling import SMOTE

In [53]:
df = pd.read_csv("cleaned_data.csv")

df.head()

,customer_id,gender,senior_citizen,partner,dependents,tenure,phone_service,multiple_lines,internet_service,online_security,...,streaming_tv,streaming_movies,contract,paperless_billing,payment_method,monthly_charges,total_charges,signup_date,churn,internal_flag
0,17267,female,0.0,No,Yes,41.0,Yes,Yes,Fiber optic,Yes,...,No,No,One year,Yes,Mailed check,53.77,848.51,2024-08-02,No,A
1,9242,Female,0.0,Yes,No,27.0,Yes,Yes,Fiber optic,Yes,...,No,No,Month-to-month,Yes,Credit card,84.76,6916.82,2022-07-15,No,A
2,19107,Male,1.0,Yes,Yes,36.0,Yes,Yes,DSL,No,...,No,Yes,Month-to-month,Y,Mailed check,28.61,1099.85,2024-07-17,No,A
3,8986,Female,0.0,No,No,1.0,N,N,No,No,...,Yes,Yes,Month-to-month,No,Electronic check,119.07,6943.98,2024-06-04,No,A
4,2867,Male,0.0,No,Yes,20.0,Yes,Yes,fiber,No,...,N,Yes,Month-to-month,No,Electronic check,108.74,5279.77,2023-04-30,No,A


In [54]:
#drop 
X = df.drop(["monthly_charges", "customer_id"], axis=1)

# Regression Target
y_regression = df["monthly_charges"]

# Classification Target
median_value = y_regression.median()

y_classification = (y_regression > median_value).astype(int)

print("Median Value:", median_value)

print(y_classification.value_counts())

# Convert date column
X["signup_date"] = pd.to_datetime(X["signup_date"])

X["signup_year"] = X["signup_date"].dt.year
X["signup_month"] = X["signup_date"].dt.month

X = X.drop("signup_date", axis=1)

# One-Hot Encoding
categorical_cols = X.select_dtypes(include="object").columns

X = pd.get_dummies(
    X,
    columns=categorical_cols,
    drop_first=True
)

Median Value: 70.26
monthly_charges
0    11452
1     8558
Name: count, dtype: int64


In [55]:
print(type(X))
print(X.shape)

<class 'pandas.core.frame.DataFrame'>
(20010, 66)


In [56]:
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X,
    y_regression,
    test_size=0.2,
    random_state=42
)

reg_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", LinearRegression())
])

reg_pipeline.fit(X_train_reg, y_train_reg)

y_pred_reg = reg_pipeline.predict(X_test_reg)

mae = mean_absolute_error(y_test_reg, y_pred_reg)

rmse = np.sqrt(mean_squared_error(y_test_reg, y_pred_reg))

r2 = r2_score(y_test_reg, y_pred_reg)

print("\nRegression Model Performance")
print("---------------------------")
print("MAE :", mae)
print("RMSE:", rmse)
print("R2 Score:", r2)


Regression Model Performance
---------------------------
MAE : 26.197149598954073
RMSE: 53.48998353897986
R2 Score: -0.0035713684742582075


In [57]:
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Ridge Regression Pipeline
ridge_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", Ridge(alpha=1.0))
])

# Train the model
ridge_pipeline.fit(X_train_reg, y_train_reg)

# Predictions
y_pred_ridge = ridge_pipeline.predict(X_test_reg)

# Evaluation
ridge_mae = mean_absolute_error(y_test_reg, y_pred_ridge)
ridge_rmse = np.sqrt(mean_squared_error(y_test_reg, y_pred_ridge))
ridge_r2 = r2_score(y_test_reg, y_pred_ridge)

print("\nRidge Regression Performance")
print("----------------------------")
print("MAE      :", ridge_mae)
print("RMSE     :", ridge_rmse)
print("R2 Score :", ridge_r2)



Ridge Regression Performance
----------------------------
MAE      : 26.19624495693966
RMSE     : 53.4896134568409
R2 Score : -0.003557481666701001


In [58]:
from sklearn.model_selection import train_test_split

X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X,
    y_classification,
    test_size=0.2,
    random_state=42,
    stratify=y_classification
)
print(X_train_clf.shape)
print(X_test_clf.shape)

(16008, 66)
(4002, 66)


In [59]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Fill missing values
imputer = SimpleImputer(strategy="median")

X_train_imputed = imputer.fit_transform(X_train_clf)
X_test_imputed = imputer.transform(X_test_clf)

# Apply SMOTE
smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train_imputed,
    y_train_clf
)

# Pipeline with Scaling + Logistic Regression
clf_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=3000, random_state=42))
])

clf_pipeline.fit(X_train_smote, y_train_smote)

# Prediction
y_pred_clf = clf_pipeline.predict(X_test_imputed)

# Evaluation

accuracy = accuracy_score(y_test_clf, y_pred_clf)

precision = precision_score(y_test_clf, y_pred_clf)

recall = recall_score(y_test_clf, y_pred_clf)

f1 = f1_score(y_test_clf, y_pred_clf)

print("\nClassification Model Performance")
print("--------------------------------")
print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)

print("\nClassification Report")
print(classification_report(y_test_clf, y_pred_clf))


Classification Model Performance
--------------------------------
Accuracy : 0.508495752123938
Precision: 0.4336283185840708
Recall   : 0.48656542056074764
F1 Score : 0.45857418111753373

Classification Report
              precision    recall  f1-score   support

           0       0.58      0.52      0.55      2290
           1       0.43      0.49      0.46      1712

    accuracy                           0.51      4002
   macro avg       0.51      0.51      0.50      4002
weighted avg       0.52      0.51      0.51      4002



In [60]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Decision Tree (Default)

dt_default = DecisionTreeClassifier(
    random_state=42
)

dt_default.fit(
    X_train_smote,
    y_train_smote
)

# Predictions

train_pred = dt_default.predict(X_train_smote)
test_pred = dt_default.predict(X_test_imputed)

# Accuracy

train_acc = accuracy_score(
    y_train_smote,
    train_pred
)

test_acc = accuracy_score(
    y_test_clf,
    test_pred
)

print("Decision Tree (Default)")
print("-----------------------")
print("Training Accuracy :", train_acc)
print("Test Accuracy     :", test_acc)

Decision Tree (Default)
-----------------------
Training Accuracy : 0.9998908535254312
Test Accuracy     : 0.5109945027486257


In [61]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Controlled Decision Tree

dt_controlled = DecisionTreeClassifier(
    max_depth=5,
    min_samples_split=20,
    random_state=42
)

dt_controlled.fit(
    X_train_smote,
    y_train_smote
)

# Predictions

train_pred = dt_controlled.predict(X_train_smote)
test_pred = dt_controlled.predict(X_test_imputed)

# Accuracy

train_acc = accuracy_score(
    y_train_smote,
    train_pred
)

test_acc = accuracy_score(
    y_test_clf,
    test_pred
)

print("Controlled Decision Tree")
print("------------------------")
print("Training Accuracy :", train_acc)
print("Test Accuracy     :", test_acc)


Controlled Decision Tree
------------------------
Training Accuracy : 0.6017245142981882
Test Accuracy     : 0.5702148925537232


In [62]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Gini Decision Tree
dt_gini = DecisionTreeClassifier(
    criterion="gini",
    max_depth=5,
    random_state=42
)

dt_gini.fit(X_train_smote, y_train_smote)

gini_pred = dt_gini.predict(X_test_imputed)

gini_acc = accuracy_score(
    y_test_clf,
    gini_pred
)


# Entropy Decision Tree
dt_entropy = DecisionTreeClassifier(
    criterion="entropy",
    max_depth=5,
    random_state=42
)

dt_entropy.fit(X_train_smote, y_train_smote)

entropy_pred = dt_entropy.predict(X_test_imputed)

entropy_acc = accuracy_score(
    y_test_clf,
    entropy_pred
)

print("Gini vs Entropy")
print("----------------")
print("Gini Accuracy   :", gini_acc)
print("Entropy Accuracy:", entropy_acc)

Gini vs Entropy
----------------
Gini Accuracy   : 0.5702148925537232
Entropy Accuracy: 0.5702148925537232


In [63]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
import pandas as pd

# Random Forest Model
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42
)

rf_model.fit(X_train_smote, y_train_smote)

# Predictions
rf_train_pred = rf_model.predict(X_train_smote)
rf_test_pred = rf_model.predict(X_test_imputed)
rf_prob = rf_model.predict_proba(X_test_imputed)[:, 1]

# Accuracy
train_acc = accuracy_score(y_train_smote, rf_train_pred)
test_acc = accuracy_score(y_test_clf, rf_test_pred)

# ROC-AUC
roc_auc = roc_auc_score(y_test_clf, rf_prob)

print("Random Forest Performance")
print("-------------------------")
print("Training Accuracy :", train_acc)
print("Test Accuracy     :", test_acc)
print("ROC-AUC           :", roc_auc)

# Feature Importance
feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

print("\nTop 5 Important Features")
print(feature_importance.head(5))


Random Forest Performance
-------------------------
Training Accuracy : 0.7115804409517572
Test Accuracy     : 0.5689655172413793
ROC-AUC           : 0.5043466616332694

Top 5 Important Features
               Feature  Importance
2        total_charges    0.049292
39     tech_support_No    0.048627
1               tenure    0.042327
27  online_security_No    0.032782
46    streaming_tv_Yes    0.032112


In [64]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

# Gradient Boosting Model
gb_model = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

# Train Model
gb_model.fit(X_train_smote, y_train_smote)

# Predictions
gb_train_pred = gb_model.predict(X_train_smote)
gb_test_pred = gb_model.predict(X_test_imputed)
gb_prob = gb_model.predict_proba(X_test_imputed)[:, 1]

# Accuracy
train_acc = accuracy_score(y_train_smote, gb_train_pred)
test_acc = accuracy_score(y_test_clf, gb_test_pred)

# ROC-AUC
roc_auc = roc_auc_score(y_test_clf, gb_prob)

print("Gradient Boosting Performance")
print("-----------------------------")
print("Training Accuracy :", train_acc)
print("Test Accuracy     :", test_acc)
print("ROC-AUC           :", roc_auc)

Gradient Boosting Performance
-----------------------------
Training Accuracy : 0.6397620606854398
Test Accuracy     : 0.5684657671164418
ROC-AUC           : 0.5071363710566053


In [65]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
import pandas as pd

# ------------------------------------
# 1. Find Bottom 5 Features
# ------------------------------------

lowest_features = feature_importance.sort_values(
    by="Importance",
    ascending=True
).head(5)

print("Lowest 5 Features")
print(lowest_features)

# ------------------------------------
# 2. Remove Bottom 5 Features
# ------------------------------------

remove_features = lowest_features["Feature"].tolist()

X_train_reduced = pd.DataFrame(
    X_train_imputed,
    columns=X.columns
).drop(columns=remove_features)

X_test_reduced = pd.DataFrame(
    X_test_imputed,
    columns=X.columns
).drop(columns=remove_features)

# ------------------------------------
# 3. Train Random Forest Again
# ------------------------------------

rf_reduced = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42
)

rf_reduced.fit(
    X_train_reduced,
    y_train_clf
)

# ------------------------------------
# 4. Compare AUC
# ------------------------------------

full_auc = roc_auc_score(
    y_test_clf,
    rf_model.predict_proba(X_test_imputed)[:, 1]
)

reduced_auc = roc_auc_score(
    y_test_clf,
    rf_reduced.predict_proba(X_test_reduced)[:, 1]
)

print("\nFeature Ablation Results")
print("------------------------")
print("Full Model AUC    :", full_auc)
print("Reduced Model AUC :", reduced_auc)


Lowest 5 Features
                   Feature  Importance
6                 gender_M    0.003768
54       contract_Two Year    0.003794
60  payment_method_E-check    0.003799
23    internet_service_Dsl    0.003844
52       contract_One Year    0.003849

Feature Ablation Results
------------------------
Full Model AUC    : 0.5043466616332694
Reduced Model AUC : 0.5011990878667918


In [66]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
import pandas as pd

# Cross Validation Strategy
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Models
models = {
    "Logistic Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=3000, random_state=42))
    ]),

    "Decision Tree": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", DecisionTreeClassifier(
            max_depth=5,
            min_samples_split=20,
            random_state=42
        ))
    ]),

    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(
            n_estimators=100,
            max_depth=10,
            random_state=42
        ))
    ]),

    "Gradient Boosting": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", GradientBoostingClassifier(
            n_estimators=100,
            learning_rate=0.1,
            random_state=42
        ))
    ])
}

results = []

for name, model in models.items():

    scores = cross_val_score(
        model,
        X,
        y_classification,
        cv=cv,
        scoring="roc_auc",
        n_jobs=-1
    )

    results.append({
        "Model": name,
        "Mean AUC": scores.mean(),
        "Std AUC": scores.std()
    })

cv_results = pd.DataFrame(results)

print("Cross Validation Results")
print(cv_results)

Cross Validation Results
                 Model  Mean AUC   Std AUC
0  Logistic Regression  0.506574  0.002355
1        Decision Tree  0.502242  0.005720
2        Random Forest  0.505173  0.007456
3    Gradient Boosting  0.505969  0.004630


In [67]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

# Pipeline
pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    RandomForestClassifier(random_state=42)
)

# Parameter Grid
param_grid = {
    "randomforestclassifier__n_estimators": [50, 100, 200],
    "randomforestclassifier__max_depth": [5, 10, None],
    "randomforestclassifier__min_samples_leaf": [1, 5]
}

# Cross Validation
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Grid Search
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1
)

# Train
grid_search.fit(X_train_clf, y_train_clf)

# Results
print("Best Parameters")
print(grid_search.best_params_)

print("\nBest CV ROC-AUC")
print(grid_search.best_score_)

Best Parameters
{'randomforestclassifier__max_depth': None, 'randomforestclassifier__min_samples_leaf': 5, 'randomforestclassifier__n_estimators': 100}

Best CV ROC-AUC
0.5087299201034077


In [68]:
import joblib

# Get the best model from GridSearchCV
best_model = grid_search.best_estimator_

# Save the model
joblib.dump(best_model, "best_model.pkl")

print("Best model saved successfully as best_model.pkl")

Best model saved successfully as best_model.pkl


In [69]:
import joblib

# Load the saved model
loaded_model = joblib.load("best_model.pkl")

print("Best model loaded successfully.")
print(loaded_model)

Best model loaded successfully.
Pipeline(steps=[('simpleimputer', SimpleImputer(strategy='median')),
                ('standardscaler', StandardScaler()),
                ('randomforestclassifier',
                 RandomForestClassifier(min_samples_leaf=5, random_state=42))])


In [70]:
import joblib

loaded_model = joblib.load("best_model.pkl")

sample_rows = X_test_clf.iloc[:2]

predictions = loaded_model.predict(sample_rows)

print(predictions)

[0 1]


In [71]:
from sklearn.metrics import roc_auc_score
import pandas as pd

best_pipeline = grid_search.best_estimator_

fractions = [0.2, 0.4, 0.6, 0.8, 1.0]

results = []

for f in fractions:

    n = int(f * len(X_train_clf))

    X_subset = X_train_clf.iloc[:n]
    y_subset = y_train_clf.iloc[:n]

    best_pipeline.fit(X_subset, y_subset)

    train_prob = best_pipeline.predict_proba(X_subset)[:, 1]
    test_prob = best_pipeline.predict_proba(X_test_clf)[:, 1]

    train_auc = roc_auc_score(y_subset, train_prob)
    test_auc = roc_auc_score(y_test_clf, test_prob)

    results.append([
        f,
        train_auc,
        test_auc
    ])

learning_curve = pd.DataFrame(
    results,
    columns=[
        "Training Fraction",
        "Training AUC",
        "Test AUC"
    ]
)

print(learning_curve)

   Training Fraction  Training AUC  Test AUC
0                0.2      0.997679  0.490338
1                0.4      0.999512  0.499534
2                0.6      0.999680  0.503761
3                0.8      0.999671  0.500927
4                1.0      0.999821  0.507617
